## 데이터 변환 연습문제
- 타입 변환, 중복/결측 처리, 파생변수, 요약변수, 정규화/표준화

**※ 지정한 데이터 파일를 기반으로 주어진 문제 해결을 위한 코드를 작성하시오.**

필요한 모듈 임포트

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns

print(f'numpy : {np.__version__}, pandas : {pd.__version__}, seaborn : {sns.__version__}')
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'

numpy : 2.5.2, pandas : 3.0.5, seaborn : 0.13.2


### 문제#1. 이커머스 주문 데이터 (order.csv)

#### 1. 파일 로딩하여 데이터프레임 `dfA`로 생성하기

In [2]:
dfA = pd.read_csv('data/order.csv')
dfA.head(2)

,order_id,customer_id,order_dt,category,unit_price,quantity,discount_rate,is_member,payment_type,gross_amount,net_amount
0,1,103,2025-05-27,Electronics,87.37,21,0.00,Y,card,1834.77,1834.77
1,2,436,2025-04-28,Beauty,114.15,45,0.05,Y,card,5136.75,4879.91


#### 2. `dfA`에서 **완전 중복 행**을 제거하고, 제거 전/후 행 수 출력하기. 중복제거한 결과는 `dfA_clean`으로 저장하기

In [3]:
dfA.shape[0]
dfA_clean = dfA.drop_duplicates()
dfA_clean.shape[0]
dfA_clean.head(2)

5050

5000

,order_id,customer_id,order_dt,category,unit_price,quantity,discount_rate,is_member,payment_type,gross_amount,net_amount
0,1,103,2025-05-27,Electronics,87.37,21,0.00,Y,card,1834.77,1834.77
1,2,436,2025-04-28,Beauty,114.15,45,0.05,Y,card,5136.75,4879.91


#### 3. `dfA`의 `net_amount` 통화 문자열을 숫자 컬럼으로 변환하기
- `dfA_clean['net_amount']`에는 `₩1,234.50` 같은 문자열이 섞여 있다.
- 이를 숫자로 변환한 `net_amount_num` 컬럼 생성하기. (변환 실패는 NaN 허용)
    - 힌트: `str.replace`, `pd.to_numeric(errors='coerce')`

In [4]:
dfA['net_amount'] = pd.to_numeric(dfA['net_amount'], errors='coerce')

In [5]:
dfA.info()

<class 'pandas.DataFrame'>
RangeIndex: 5050 entries, 0 to 5049
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   order_id       5050 non-null   int64  
 1   customer_id    5050 non-null   int64  
 2   order_dt       5050 non-null   str    
 3   category       5050 non-null   str    
 4   unit_price     4908 non-null   float64
 5   quantity       5050 non-null   int64  
 6   discount_rate  4759 non-null   float64
 7   is_member      5050 non-null   str    
 8   payment_type   4842 non-null   str    
 9   gross_amount   5050 non-null   float64
 10  net_amount     4456 non-null   float64
dtypes: float64(4), int64(3), str(4)
memory usage: 434.1 KB


#### 4. dfA의 결측치 처리하기(수치/범주 혼합)
- `unit_price` 결측: **category별 중앙값**으로 대체
- `discount_rate` 결측: 0으로 대체
- `payment_type` 결측: 최빈값으로 대체
- 결과를 `dfA_clean`에 반영하기

In [6]:
#dfA.groupby('category')['unit_price'].transform('median')
dfA_clean['unit_price'] = dfA['unit_price'].fillna(dfA.groupby('category')['unit_price'].transform('median'))
#dfA['unit_price'].isna().sum()
dfA_clean['discount_rate'] = dfA['discount_rate'].fillna(0)
dfA_clean['payment_type'] = dfA['payment_type'].fillna(
    dfA.groupby('category')['payment_type'].transform(
                                                    lambda x: x.mode()[0]
    )
)


In [7]:
dfA_clean
dfA_clean['unit_price'].isna().sum()
dfA_clean['discount_rate'].isna().sum()
dfA_clean['payment_type'].isna().sum()

,order_id,customer_id,order_dt,category,unit_price,quantity,discount_rate,is_member,payment_type,gross_amount,net_amount
0,1,103,2025-05-27,Electronics,87.37,21,0.00,Y,card,1834.77,1834.77
1,2,436,2025-04-28,Beauty,114.15,45,0.05,Y,card,5136.75,4879.91
2,3,271,2025-06-01,Fashion,99.25,38,0.10,Y,card,3771.50,3394.35
3,4,107,2025-03-13,Grocery,74.93,20,0.15,Y,card,1726.20,"₩1,467.27"
4,5,72,2025-12-29,Fashion,63.58,62,0.05,Y,card,3941.96,3744.86
...,...,...,...,...,...,...,...,...,...,...,...
4995,4996,679,2025-11-01,Electronics,72.32,82,0.10,Y,bank,6503.42,5853.08
4996,4997,247,2025-08-13,Beauty,49.83,93,0.05,N,card,4634.19,"₩4,402.48"
4997,4998,559,2025-11-03,Fashion,41.15,78,0.00,N,card,3209.70,3049.21
4998,4999,758,2025-07-07,Electronics,68.86,29,0.15,Y,card,1996.94,1697.4


np.int64(0)

np.int64(0)

np.int64(0)

#### 5. 4번에서 생성한 dfA_clean에서 날짜/매출 관련 파생변수 생성하기
- `order_dt`에서 `year`, `month`, `dow` 컬럼 생성 (dow : day_of_week)
- `dow`컬럼을 이용하여 토/일=1, 그 외=0를 갖는 `is_weekend` 컬럼 생성
- 단가 `unit_price`와 수량 `quantity`을 이용하여 총액 `gross_amount_calc` 컬럼 생성
- `gross_amount_calc`와 `discount_rate`컬럼을 이용하여 계산한 `net_amount_calc`컬럼 생성 : 할인률 공제한 금액

In [8]:
dfA_clean['order_dt'] = dfA_clean['order_dt'].astype('datetime64[ns]')
dfA_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   order_id       5000 non-null   int64         
 1   customer_id    5000 non-null   int64         
 2   order_dt       5000 non-null   datetime64[ns]
 3   category       5000 non-null   str           
 4   unit_price     5000 non-null   float64       
 5   quantity       5000 non-null   int64         
 6   discount_rate  5000 non-null   float64       
 7   is_member      5000 non-null   str           
 8   payment_type   5000 non-null   str           
 9   gross_amount   5000 non-null   float64       
 10  net_amount     5000 non-null   str           
dtypes: datetime64[ns](1), float64(3), int64(3), str(4)
memory usage: 429.8 KB


In [9]:
dfA_clean['year'] = dfA_clean['order_dt'].dt.year
dfA_clean['month'] = dfA_clean['order_dt'].dt.month
dfA_clean['dow'] = dfA_clean['order_dt'].dt.dayofweek

In [10]:
dfA_clean['is_weekend'] = dfA_clean['dow'].apply(lambda x: 0 if (x<=4) else 1)

In [11]:
dfA_clean['is_weekend']

0       0
1       0
2       1
3       0
4       0
       ..
4995    1
4996    0
4997    0
4998    0
4999    0
Name: is_weekend, Length: 5000, dtype: int64

In [12]:
dfA_clean.head(2)

,order_id,customer_id,order_dt,category,unit_price,quantity,discount_rate,is_member,payment_type,gross_amount,net_amount,year,month,dow,is_weekend
0,1,103,2025-05-27,Electronics,87.37,21,0.00,Y,card,1834.77,1834.77,2025,5,1,0
1,2,436,2025-04-28,Beauty,114.15,45,0.05,Y,card,5136.75,4879.91,2025,4,0,0


#### 6. 고객 요약변수 생성 후 원본에 결합하기
- 고객별 `order_cnt`, `net_sum`, `net_mean`, `recency_days` 생성
- `recency_days`는 **전체 최대 주문일 - 고객별 마지막 주문일** (일 단위)
- 결과 요약을 dfA_clean에 merge 하기.

In [14]:
# dfA_clean.drop(columns = ['order_cnt','net_sum','net_mean','recency_days'])
# dfA_clean.info()
dfA_customer = pd.DataFrame()
dfA_customer['customer_id'] = dfA_clean['customer_id']
dfB = dfA_clean.copy()

In [18]:
dfA_customer['order_cnt'] = dfA_clean.groupby('customer_id')['order_id'].transform('size')
dfA_customer['net_sum'] = dfA_clean.groupby('customer_id')['quantity'].transform('sum')
dfA_customer['net_mean'] = dfA_clean.groupby('customer_id')['quantity'].transform('mean')
max_day = dfA_clean['order_dt'].max()
dfA_customer['recency_days'] = (max_day - dfA_clean.groupby('customer_id')['order_dt'].transform('max')).dt.days

dfB = pd.merge(dfA_clean, dfA_customer, on='customer_id', how='left')

In [19]:
dfA_customer.head()
dfB

,customer_id,order_cnt,net_sum,net_mean,recency_days
0,103,5,267,53.400000,3
1,436,9,474,52.666667,29
2,271,6,300,50.000000,7
3,107,5,247,49.400000,180
4,72,4,258,64.500000,2


,order_id,customer_id,order_dt,category,unit_price,quantity,discount_rate,is_member,payment_type,gross_amount,net_amount,year,month,dow,is_weekend,order_cnt,net_sum,net_mean,recency_days
0,1,103,2025-05-27,Electronics,87.37,21,0.00,Y,card,1834.77,1834.77,2025,5,1,0,5,267,53.400000,3
1,1,103,2025-05-27,Electronics,87.37,21,0.00,Y,card,1834.77,1834.77,2025,5,1,0,5,267,53.400000,3
2,1,103,2025-05-27,Electronics,87.37,21,0.00,Y,card,1834.77,1834.77,2025,5,1,0,5,267,53.400000,3
3,1,103,2025-05-27,Electronics,87.37,21,0.00,Y,card,1834.77,1834.77,2025,5,1,0,5,267,53.400000,3
4,1,103,2025-05-27,Electronics,87.37,21,0.00,Y,card,1834.77,1834.77,2025,5,1,0,5,267,53.400000,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
36791,5000,127,2025-03-11,Grocery,107.70,94,0.05,N,card,10123.80,9617.61,2025,3,1,0,9,461,51.222222,24
36792,5000,127,2025-03-11,Grocery,107.70,94,0.05,N,card,10123.80,9617.61,2025,3,1,0,9,461,51.222222,24
36793,5000,127,2025-03-11,Grocery,107.70,94,0.05,N,card,10123.80,9617.61,2025,3,1,0,9,461,51.222222,24
36794,5000,127,2025-03-11,Grocery,107.70,94,0.05,N,card,10123.80,9617.61,2025,3,1,0,9,461,51.222222,24


#### 7. 수치 컬럼 정규화/표준화하기

- 대상: `unit_price, quantity, net_amount_calc, order_cnt, net_sum, recency_days`
- Min-Max 결과는 `*_minmax`, Z-score 결과는 `*_z`로 컬럼 생성

### 문제 #2. 학습/평가 데이터 (students.csv)

#### 1. students.csv 파일 로딩하여  데이터프레임 dfC 생성

#### 2. 중복 제거 및 attendance_rate 변환

- dfC 중복 제거 후 `dfC_clean` 생성
- `attendance_rate`의 `"92.0%"`를 0~1 실수로 변환
- 변환 후 1보다 크면 100으로 나누기

#### 3. 결측치 처리 : 그룹 대체

- `mid_score`, `final_score`: grade별 중앙값
- `study_hours`: region별 중앙값
- `attendance_rate`: 전체 중앙값
- 모든 대상 컬럼을 numeric으로 강제 변환하기

#### 4. 파생변수 생성하기 : avg_score, score_gap, is_pass, join_year/month

- `avg_score` = (mid+final)/2
- `score_gap` = final - mid
- `is_pass` = (avg_score >= 60) & (attendance_rate >= 0.85) -> 0/1
- join_dt에서 `join_year`, `join_month` 생성

#### 5. 지역-학년 요약변수 결합 + 정규화/표준화

- (region, grade)별로
  - `region_grade_avg`(avg_score 평균)
  - `region_grade_std`(avg_score 표준편차)
  - `pass_rate`(is_pass 평균)
  - `cnt`(학생 수)
- dfC_clean에 merge
- 수치 대상: `study_hours, attendance_rate, mid_score, final_score, avg_score, score_gap, region_grade_avg, pass_rate`
  - Min-Max: `*_minmax`, Z-score: `*_z`

### 문제 #3. IoT 센서 시계열 데이터 (sensor.csv)

#### 1. sensor.csv 파일 로딩하야 데이터프레임 dfB 생성하기

#### 2. 중복 제거 및 temp_c 타입 변환하기

- dfB의 중복 제거 후 `dfB_clean` 컬럼 생성
- `temp_c` 컬럼에 `'23.5C'`와 같은 데이터들을 숫자로 변환(변환 실패는 NaN)
- `ts` 기준으로 sensor_id별 정렬까지 수행

#### 3. 결측치 보간 + 잔여 결측 처리

- `hum_pct`, `vibration`을 sensor_id 그룹별로 **선형 보간**(양방향)
- 이후에도 남은 결측은 sensor_id별 중앙값으로 대체

#### 4. 6시간 롤링 요약변수 생성

- sensor_id별 `temp_roll6_mean`(6시간 이동평균)
- sensor_id별 `vib_roll6_std`(6시간 이동표준편차, NaN은 0으로)

#### 5. 일 단위 요약변수 생성 후 결합하기

- `day = ts.date` 생성
- (sensor_id, day)별로
  - `temp_day_mean`(평균)
  - `temp_day_max`(최대)
  - `hum_day_mean`(평균)
  - `vib_day_p95`(95퍼센타일)
- 결과를 dfB_clean에 merge

#### 6. 정규화/표준화 컬럼 생성하기

- 대상: `temp_c, hum_pct, vibration, temp_roll6_mean, vib_roll6_std, temp_day_mean, vib_day_p95`
- Min-Max: `*_minmax`, Z-score: `*_z`

---